# P5 · Đánh giá & tổng hợp — Bước 6–7–8

**Input (đọc):** data/outputs/forecasts/*.parquet (từ P3/P4)

**Output (bạn phải tạo, đúng format):** bảng RMSSE/SPL/coverage theo model×nhóm, bảng thắng (DM/MCB), calibration

**Code mẫu chạy được:** mở `notebooks/M5_reference.ipynb` phần **§7** — copy các cell sang đây rồi mở rộng.

> Quy tắc vàng: không sửa `common/`; không shuffle; tôn trọng trục thời gian; ML dùng lag ≥ 28. Forecast lưu vào `data/outputs/forecasts/<model>.parquet` với cột: `series_id,date,d_int,quantile_level,value,model`.

## Cell setup dùng chung (chạy trước) — nạp tham số cố định + dữ liệu
Các cell dưới đây GIỐNG NHAU ở mọi notebook, đảm bảo cả nhóm dùng chung 1 cấu hình & dữ liệu.

In [ ]:
import sys, subprocess, importlib, time, warnings, os, json, math, random
# --- 0.2 Import lõi + dò package nào dùng được ------------------------------
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42); random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4); plt.rcParams["axes.grid"] = True

def can_import(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

AVAILABLE = {
    "pyarrow":      can_import("pyarrow"),
    "lightgbm":     can_import("lightgbm"),
    "statsforecast":can_import("statsforecast"),
    "chronos":      can_import("chronos"),
    "torch":        can_import("torch"),
    "ttm":          can_import("tsfm_public"),
}
try:
    import torch
    AVAILABLE["cuda"] = torch.cuda.is_available()
except Exception:
    AVAILABLE["cuda"] = False

print("Package/phần cứng dùng được:")
for k, v in AVAILABLE.items():
    print(f"  {'✓' if v else '✗'} {k}: {v}")
PARQUET_ENGINE = "pyarrow" if AVAILABLE["pyarrow"] else None

In [ ]:
# --- Nạp THAM SỐ CỐ ĐỊNH từ common/config.yaml (nguồn chân lý duy nhất) -------
import yaml
from pathlib import Path
_c = yaml.safe_load(open("common/config.yaml", encoding="utf-8"))
CONFIG = dict(
    SEED=_c["seed"], HORIZON=_c["horizon"],
    SAMPLE_PER_SEGMENT=_c.get("sampling", {}).get("cap", 1000),
    TRAIN_END=_c["split"]["train_end"], VAL_END=_c["split"]["val_end"],
    TEST_END=_c["split"]["test_end"],
    QUANTILES=_c["quantiles"], ADI_TH=_c["sbc"]["adi_threshold"],
    CV2_TH=_c["sbc"]["cv2_threshold"],
    CHRONOS_MODEL=_c["foundation"]["chronos_model"],
    CHRONOS_CONTEXT=_c["foundation"]["chronos_context"],
    TTM_MODEL=_c["foundation"]["ttm_model"],
    TTM_CONTEXT=_c["foundation"]["ttm_context"],
)
CONFIG["TEST_START"] = CONFIG["VAL_END"] + 1
Q = CONFIG["QUANTILES"]
RAW = Path("data/raw"); OUT = Path("data/outputs"); FC = OUT / "forecasts"
OUT.mkdir(parents=True, exist_ok=True); FC.mkdir(parents=True, exist_ok=True)
PARQUET_ENGINE = "pyarrow"
def save_parquet(df, path):
    path = Path(path); df.to_parquet(path, index=False); return path
print("CONFIG nạp từ common/config.yaml:", {k: CONFIG[k] for k in ["SEED","HORIZON","TRAIN_END","VAL_END","TEST_END"]})

In [ ]:
# --- Nạp DỮ LIỆU CHUNG: long (P1 đã sinh) + mẫu + nhãn + calendar/giá ---------
import numpy as np, pandas as pd
calendar = pd.read_csv(RAW / "calendar.csv")
prices   = pd.read_csv(RAW / "sell_prices.csv")

_long_path = Path("data/processed/data_long.parquet")
assert _long_path.exists(), ("Chưa có data_long. Hãy chạy notebooks/P1.ipynb trước "
                             "(hoặc lấy từ người P1) để sinh data/processed/data_long.parquet.")
long = pd.read_parquet(_long_path)
long["date"] = pd.to_datetime(long["date"])

sample_ids = pd.read_csv("common/sample_ids.csv")["series_id"].tolist()
seg_labels = pd.read_parquet("common/segment_labels.parquet")

# schema ngày (train/val/test) cho cửa sổ gốc
schema = dict(
    train_days=[f"d_{i}" for i in range(1, CONFIG["TRAIN_END"]+1)],
    val_days=[f"d_{i}" for i in range(CONFIG["TRAIN_END"]+1, CONFIG["VAL_END"]+1)],
    test_days=[f"d_{i}" for i in range(CONFIG["VAL_END"]+1, CONFIG["TEST_END"]+1)],
)
# insample (<= val_end) dùng cho model cổ điển & FM
insample = long[long.d_int <= CONFIG["VAL_END"]].sort_values(["series_id","d_int"])
print(f"long: {long.shape} | mẫu: {len(sample_ids)} chuỗi | nhãn: {seg_labels.shape}")

In [ ]:
# --- 2.5 Bảng actuals cho test (dùng cho đánh giá) --------------------------
# actual test: các ngày d_TEST_START..d_TEST_END của chuỗi mẫu
test_days = schema["test_days"]
actual_test = (long[long.d.isin(test_days)]
               [["series_id","date","d_int","sales","segment"]]
               .rename(columns={"sales":"actual"})
               .reset_index(drop=True))
# train history (long) để tính scale RMSSE/pinball
train_hist = long[long.d_int <= CONFIG["VAL_END"]][["series_id","d_int","sales"]].copy()
print("actual_test:", actual_test.shape, "| train_hist:", train_hist.shape)

In [ ]:
# --- 3.1 Tiện ích scale (RMSSE & pinball) + trọng số theo chuỗi -------------
def _scales(y):
    '''scale MSE & MAE cua naive-1 tren insample, tinh tu lan ban dau tien (kieu M5).'''
    nz = np.nonzero(y)[0]
    if len(nz) < 2:
        return np.nan, np.nan
    y = y[nz[0]:]
    d = np.diff(y)
    if len(d) == 0:
        return np.nan, np.nan
    return float(np.mean(d**2)), float(np.mean(np.abs(d)))

scale_rmsse, scale_mae = {}, {}
for sid, g in train_hist.sort_values("d_int").groupby("series_id"):
    s2, s1 = _scales(g["sales"].to_numpy(dtype=np.float64))
    scale_rmsse[sid], scale_mae[sid] = s2, s1

# trọng số WSPL = doanh số 28 ngày insample cuối (kiểu M5 uncertainty)
last28 = long[(long.d_int > CONFIG["VAL_END"]-28) & (long.d_int <= CONFIG["VAL_END"])].copy()
last28["dollar"] = last28["sales"] * last28["sell_price"].fillna(0)
weight = last28.groupby("series_id")["dollar"].sum()
weight = (weight / weight.sum()).to_dict()   # chuẩn hoá tổng = 1
print(f"Đã tính scale cho {len(scale_rmsse)} chuỗi. Ví dụ scale_mae: "
      f"{list(scale_mae.items())[0]}")

In [ ]:
# --- 3.2 Dispersion NB theo chuỗi (từ insample) -----------------------------
mv = train_hist.groupby("series_id")["sales"].agg(["mean","var"])
def _r(row):
    m, v = row["mean"], row["var"]
    if v > m and m > 0:
        return m*m / (v - m)      # r hữu hạn -> NB
    return np.inf                 # -> Poisson
disp_r = mv.apply(_r, axis=1).to_dict()

def point_to_quantiles(point_df, name):
    '''point_df: series_id,date,d_int,value(mean) -> long 9 quantile theo NB/Poisson.'''
    out = []
    pf = point_df.copy()
    pf["value"] = pf["value"].clip(lower=0).astype(float)
    pf["r"] = pf["series_id"].map(disp_r).fillna(np.inf)
    mu = pf["value"].to_numpy()
    r  = pf["r"].to_numpy()
    finite = np.isfinite(r) & (r > 0)
    for u in Q:
        qv = np.empty_like(mu)
        # NB: mean=mu, size=r  -> p = r/(r+mu)
        if finite.any():
            rr = r[finite]; mm = mu[finite]
            p = rr / (rr + np.where(mm > 0, mm, 1e-9))
            qv[finite] = stats.nbinom.ppf(u, rr, p)
        if (~finite).any():
            qv[~finite] = stats.poisson.ppf(u, mu[~finite])
        block = pf[["series_id","date","d_int"]].copy()
        block["quantile_level"] = u
        block["value"] = np.nan_to_num(qv, nan=0.0)
        out.append(block)
    res = pd.concat(out, ignore_index=True)
    res["model"] = name
    return res[["series_id","date","d_int","quantile_level","value","model"]]
print("Sẵn sàng point_to_quantiles(). Ví dụ disp_r:", list(disp_r.items())[0])

In [ ]:
# --- 3.3 evaluate(): RMSSE, SPL/WSPL, coverage theo từng chuỗi --------------
COV_PAIRS = [(0.25,0.75,0.50),(0.165,0.835,0.67),(0.025,0.975,0.95),(0.005,0.995,0.99)]

def evaluate(fdf, name):
    '''fdf: long forecast (series_id,d_int,quantile_level,value). Tra per-series metrics.'''
    piv = (fdf.pivot_table(index=["series_id","d_int"], columns="quantile_level", values="value")
              .reset_index())
    m = piv.merge(actual_test[["series_id","d_int","actual","segment"]],
                  on=["series_id","d_int"], how="inner")
    rows = []
    for sid, g in m.groupby("series_id"):
        a = g["actual"].to_numpy(dtype=float)
        p50 = g[0.5].to_numpy(dtype=float) if 0.5 in g else np.full_like(a, np.nan)
        sr, sm = scale_rmsse.get(sid, np.nan), scale_mae.get(sid, np.nan)
        rmsse = np.sqrt(np.mean((a-p50)**2)/sr) if (sr and sr > 0) else np.nan
        spls = []
        for u in Q:
            if u in g:
                q = g[u].to_numpy(dtype=float)
                pin = np.mean(np.maximum(u*(a-q), (u-1)*(a-q)))
                spls.append(pin/sm if (sm and sm > 0) else np.nan)
        spl = np.nanmean(spls) if spls else np.nan
        covs = {}
        for lo, hi, nom in COV_PAIRS:
            if lo in g and hi in g:
                inside = ((a >= g[lo].to_numpy()) & (a <= g[hi].to_numpy())).mean()
                covs[f"cov{int(nom*100)}"] = inside
        rows.append(dict(series_id=sid, segment=g["segment"].iloc[0], model=name,
                         RMSSE=rmsse, SPL=spl, weight=weight.get(sid, np.nan), **covs))
    return pd.DataFrame(rows)
print("evaluate() sẵn sàng.")

## Việc của P5 (đọc forecast từ file → chấm điểm → tổng hợp)

In [ ]:
# --- P5: đọc MỌI forecast từ file (P3/P4 đã tạo) rồi tính metric -------------
FCDIR = Path("data/outputs/forecasts")
all_fc = {p.stem: pd.read_parquet(p) for p in sorted(FCDIR.glob("*.parquet"))}
assert all_fc, "Chưa có forecast nào trong data/outputs/forecasts/. Chờ P3/P4 chạy xong."
print("Model có forecast:", list(all_fc.keys()))
metrics = pd.concat([evaluate(df, nm) for nm, df in all_fc.items()], ignore_index=True)
metrics = metrics.dropna(subset=["segment"])
print("metrics:", metrics.shape)

In [ ]:
# --- 7.2 Bảng tổng hợp theo model × nhóm -----------------------------------
SEG_ORDER = ["smooth","erratic","intermittent","lumpy"]

def _wmean(s, w):
    w = w.reindex(s.index); ok = s.notna() & w.notna() & (w > 0)
    return np.average(s[ok], weights=w[ok]) if ok.any() else np.nan

agg = (metrics.groupby(["model","segment"])
       .apply(lambda g: pd.Series({
           "RMSSE": g.RMSSE.mean(),
           "SPL":   g.SPL.mean(),
           "WSPL":  _wmean(g.set_index("series_id").SPL, g.set_index("series_id").weight),
           "cov95": g["cov95"].mean() if "cov95" in g else np.nan,
       }))
       .reset_index())

rmsse_tab = agg.pivot(index="model", columns="segment", values="RMSSE")[SEG_ORDER]
spl_tab   = agg.pivot(index="model", columns="segment", values="SPL")[SEG_ORDER]
cov_tab   = agg.pivot(index="model", columns="segment", values="cov95")[SEG_ORDER]

print("=== RMSSE (điểm; thấp = tốt) theo model × nhóm ==="); display(rmsse_tab.round(3))
print("\\n=== SPL (xác suất; thấp = tốt) theo model × nhóm ==="); display(spl_tab.round(3))
print("\\n=== Coverage 95% (mục tiêu 0.95) theo model × nhóm ==="); display(cov_tab.round(3))
save_parquet(agg, OUT / "metrics_by_model_segment.parquet")

In [ ]:
# --- 7.3 Model thắng mỗi nhóm (theo SPL, có ghi chú mức chênh) --------------
def _winner(col):
    s = col.dropna().sort_values()
    if len(s) == 0: return pd.Series({"winner":"-","best":np.nan,"runner_up":"-","margin_%":np.nan,"verdict":"-"})
    best, bval = s.index[0], s.iloc[0]
    if len(s) > 1:
        run, rval = s.index[1], s.iloc[1]
        margin = 100*(rval - bval)/bval if bval > 0 else np.nan
    else:
        run, margin = "-", np.nan
    verdict = "tương đương" if (pd.notna(margin) and margin < 5) else "thắng rõ"
    return pd.Series({"winner":best,"best":round(bval,3),"runner_up":run,
                      "margin_%":round(margin,1) if pd.notna(margin) else np.nan,"verdict":verdict})

win_spl = spl_tab.apply(_winner).T
print("=== MODEL THẮNG MỖI NHÓM (tiêu chí SPL / xác suất) ===")
display(win_spl)
print("\\n=== (tham chiếu) model thắng theo RMSSE ===")
display(rmsse_tab.apply(_winner).T[["winner","best","margin_%","verdict"]])

In [ ]:
# --- 7.4 Bảng quyết định 'nhu cầu nào -> model nào' ------------------------
decision = win_spl[["winner","verdict","margin_%"]].copy()
decision.columns = ["Nên dùng (SPL)", "Độ tin cậy", "Chênh vs á quân (%)"]
decision.index.name = "Nhóm demand"
print("╔═══════════════════ BẢNG QUYẾT ĐỊNH ═══════════════════╗")
display(decision)
save_parquet(decision.reset_index(), OUT / "decision_table.parquet")

In [ ]:
# --- 7.5 Error analysis: bias (P50 - actual) theo model × nhóm -------------
bias_rows = []
for nm, df in all_fc.items():
    p50 = df[df.quantile_level == 0.5][["series_id","d_int","value"]]
    mrg = p50.merge(actual_test[["series_id","d_int","actual","segment"]], on=["series_id","d_int"])
    mrg["bias"] = mrg["value"] - mrg["actual"]
    b = mrg.groupby("segment")["bias"].mean().reindex(SEG_ORDER)
    b.name = nm; bias_rows.append(b)
bias_tab = pd.concat(bias_rows, axis=1).T
print("=== Bias trung bình (P50 - actual); >0 = over-forecast ===")
display(bias_tab.round(3))

In [ ]:
# --- 7.6 Biểu đồ forecast vs actual: 1 chuỗi đại diện mỗi nhóm -------------
prefer = [m for m in ["lgbm","chronos_bolt","sba","snaive","naive"] if m in all_fc][:1]
model_for_plot = prefer[0] if prefer else list(all_fc)[0]
fdf = all_fc[model_for_plot]
fig, axes = plt.subplots(2, 2, figsize=(13, 8)); axes = axes.ravel()
for ax, seg in zip(axes, SEG_ORDER):
    cand = actual_test[actual_test.segment == seg].groupby("series_id")["actual"].sum()
    cand = cand[cand > 0]
    if len(cand) == 0:
        ax.set_title(f"{seg}: (không có chuỗi bán trong test)"); continue
    sid = cand.index[len(cand)//2]
    hist = long[(long.series_id == sid) & (long.d_int > CONFIG["VAL_END"]-56)
                & (long.d_int <= CONFIG["VAL_END"])]
    act  = actual_test[actual_test.series_id == sid].sort_values("d_int")
    f    = fdf[fdf.series_id == sid]
    pv   = f.pivot_table(index="d_int", columns="quantile_level", values="value").sort_index()
    ax.plot(hist.d_int, hist.sales, color="gray", lw=1, label="lịch sử")
    ax.plot(act.d_int, act.actual, "k.-", lw=1, label="actual")
    ax.plot(pv.index, pv[0.5], "b-", lw=1.2, label="P50")
    if 0.165 in pv and 0.835 in pv:
        ax.fill_between(pv.index, pv[0.165], pv[0.835], color="b", alpha=0.2, label="P17–P83")
    ax.set_title(f"{seg} · {sid}"); ax.legend(fontsize=7)
fig.suptitle(f"Forecast vs Actual — mô hình: {model_for_plot}"); plt.tight_layout(); plt.show()